# Evaluation of temporal structure

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import functools
import IPython
import math
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
from statsmodels.tsa.stattools import acf
import string
import xarray as xr

from mlde_analysis.furflex_data import prep_eval_data
from mlde_analysis.psd import plot_psd, pysteps_rapsd
from mlde_analysis.display import pretty_table

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

## Figure: structure

* acf

In [ ]:
nlags = 23

In [ ]:
%%time

def func(da, nlags=23):
    # display(da)
    return xr.apply_ufunc(
        acf,
        da.squeeze("date").compute(),
        input_core_dims=[["hour"]],
        output_core_dims=[["lag"]],
        vectorize=True,
        kwargs=dict(nlags=nlags),
        # dask="allowed",
        # dask="parallelized",
        # dask_gufunc_kwargs={"output_sizes": {"lag": nlags+1}},
    )

# EVAL_DS["CPM"]["target_pr"].drop_vars(["time_period", "dec_adjusted_year", "stratum", "tp_season_year"]) \
#     .coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(func).compute()

acf_da = EVAL_DS["CPM"]["pred_pr"].drop_vars(["time_period", "dec_adjusted_year", "stratum", "tp_season_year"]) \
    .coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(func, nlags=nlags).compute()
acf_da

In [ ]:
%%time

def full_dask_acf(da, nlags = 23):
    # display(da)
    return xr.apply_ufunc(
        acf,
        da.squeeze("date"),
        input_core_dims=[["hour"]],
        output_core_dims=[["lag"]],
        vectorize=True,
        kwargs=dict(nlags=nlags),
        # dask="allowed",
        dask="parallelized",
        dask_gufunc_kwargs={"output_sizes": 
            {
                "lag": nlags+1,
                # "ensemble_member": da["ensemble_member"].size,
                # "model": da["model"].size,
                # "sample_id": da["sample_id"].size,
                # "grid_latitude": da["grid_latitude"].size,
                # "grid_longitude": da["grid_longitude"].size,
            }
        },
    )
# EVAL_DS["CPM"]["target_pr"].drop_vars(["time_period", "dec_adjusted_year", "stratum", "tp_season_year"]) \
#     .coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(func).compute()

full_dask_acf_da = EVAL_DS["CPM"]["pred_pr"].drop_vars(["time_period", "dec_adjusted_year", "stratum", "tp_season_year"]) \
    .coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(full_dask_acf, nlags=nlags).compute() #.mean(["sample_id", "ensemble_member", "date"]).compute()
full_dask_acf_da

In [ ]:
error_threshold = 1e-6
(np.abs(acf_da - full_dask_acf_da) > error_threshold).sum().item(), full_dask_acf_da.size - (np.abs(acf_da - full_dask_acf_da) <= error_threshold).sum().item()

In [ ]:
np.isnan(full_dask_acf_da).sum().item(), np.isnan(acf_da).sum().item(), (np.isnan(full_dask_acf_da) == np.isnan(acf_da)).all().item()

In [ ]:
(acf(
    EVAL_DS["CPM"]["pred_pr"].drop_vars(
        ["time_period", "dec_adjusted_year", "stratum", "tp_season_year"]
    ).isel(
        ensemble_member=1, sample_id=0, time=slice(24*5, 24*6), grid_longitude=32, grid_latitude=32, model=0
    ).compute(),
    nlags=nlags,
) == full_dask_acf_da.isel(ensemble_member=1, sample_id=0, date=5, grid_longitude=32, grid_latitude=32, model=0)).all().item()

In [ ]:
full_dask_acf_da = full_dask_acf_da.assign_coords(lag=np.arange(0,24))
full_dask_acf_da.mean(["date", "ensemble_member", "sample_id"]).drop_sel(lag=0).isel(model=0).plot(col="lag", col_wrap=8)

In [ ]:
client.close()